# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

/var/folders/hk/7jjnrbrj53n1t8_bmkhf0_k00000gn/T/ipykernel_93002/3307975217.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 3
Very Important: Please Confirm the Iteration Number is Iteration 3
Very Important: Please Confirm the Iteration Number is Iteration 3


In [3]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

[WARNING 07-02 10:22:24] ax.modelbridge.transforms.standardize_y: Outcome surfactant_input is constant, within tolerance.


**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  3

**************************************************************************************************************


KeyboardInterrupt: 

# process results

In [4]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 47, 's2': 60, 's3': 50, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 100})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 74, 's2': 0, 's3': 88, 's4': 66, 's5': 22, 's6': 52, 's7': 58, 's8': 79, 'surfactant_conc': 35, 'drug_conc': 100})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 78, 's2': 88, 's3': 1, 's4': 3, 's5': 32, 's6': 94, 's7': 44, 's8': 58, 'surfactant_conc': 1, 'drug_conc': 100})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', paramete

In [5]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [6]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: G7
Deep plate will start at: B1

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [7]:
hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_3.py


In [8]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,0
1,1,1
2,2,0


In [9]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_input,drug_conc,success,complexity
0,9,0,0,100,100,0,0,0,0,0.5,25.0,0,2
1,10,0,0,0,0,0,100,100,0,50.0,25.0,1,2
2,11,0,0,0,0,0,100,100,0,0.5,25.0,0,2


In [10]:
norm_results = hf.normalize_data(results, 'normalize')

In [11]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_input,drug_conc,success,complexity
0,9,0,0,100,100,0,0,0,0,0.01,1.0,0.0,0.25
1,10,0,0,0,0,0,100,100,0,1.00,1.0,1.0,0.25
2,11,0,0,0,0,0,100,100,0,0.01,1.0,0.0,0.25


# load the results to the optimizer

In [12]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-27 13:31:02] ax.service.ax_client: Completed trial 9 with data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)}.
[INFO 06-27 13:31:02] ax.service.ax_client: Completed trial 10 with data: {'success': (1.0, None), 'surfactant_input': (1.0, None), 'complexity': (0.25, None)}.
[INFO 06-27 13:31:02] ax.service.ax_client: Completed trial 11 with data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)}.


Trial 9: success=0 → overriding surfactant_input & complexity to 1
Trial 10: success=1 → using surfactant_input=1.0, complexity=0.25
Trial 11: success=0 → overriding surfactant_input & complexity to 1


AxClient(experiment=Experiment(drug_surfactant))